In [1]:
import pandas as pd
import pickle
import json

from pyboolnet.external.bnet2primes import bnet_file2primes
from pystablemotifs.format import primes2bnet

In [ ]:
# choose behavior here
AMBIGUOUS_MODE = "positive"   # or "negative"

INPUT_FILE = "edges.csv"

json_file = "../case_study/T_cell/Tcell_config.json"

INTERMEDIATE_FILE = "Tcell_merged_raw.bnet"
OUTPUT_CACHE = "Tcell_primes.pkl"
OUTPUT_FILE = "Tcell_merged.bnet"

In [3]:
df = pd.read_csv(INPUT_FILE)

allowed_signs = {"positive", "negative", "ambiguous"}

invalid = df.loc[~df["Sign"].isin(allowed_signs), "Sign"].unique()
if len(invalid) > 0:
    raise ValueError(
        f"Invalid values in 'Sign': {invalid}. Allowed: {allowed_signs}"
    )

In [4]:
f = open(json_file)
json_dict = json.load(f)

CONSTRAINTS = json_dict["constraints"]

In [5]:
def build_rule(group, constraints, ambiguous_as="positive"):
    """
    ambiguous_as: "positive" or "negative"
    constraints: {"regulate": {target: [regulators]}, "necessary": {target: [regulators]}}
    """
    target = group.name
    actual_regulators = set(group["Regulator"])

    if ambiguous_as not in {"positive", "negative"}:
        raise ValueError("ambiguous_as must be 'positive' or 'negative'")

    # 1. Handle "regulate" constraint validation
    regulate_constraints = constraints.get("regulate", {})
    required_regulators = set(regulate_constraints.get(target, []))
    if not required_regulators.issubset(actual_regulators):
        raise ValueError(f"Missing required 'regulate' regulators for target {target}")

    # 2. Handle "necessary" constraint validation
    necessary_constraints = constraints.get("necessary", {})
    necessary_regulators = set(necessary_constraints.get(target, []))
    if not necessary_regulators.issubset(actual_regulators):
        raise ValueError(f"Missing required 'necessary' regulators for target {target}")

    # 3. Separate regulators by sign
    pos = group.loc[group["Sign"] == "positive", "Regulator"].tolist()
    neg = group.loc[group["Sign"] == "negative", "Regulator"].tolist()
    amb = group.loc[group["Sign"] == "ambiguous", "Regulator"].tolist()

    if ambiguous_as == "positive":
        pos += amb
    else:
        neg += amb

    # 4. Filter out non-necessary regulators for the base rule
    base_pos = [r for r in pos if r not in necessary_regulators]
    base_neg = [r for r in neg if r not in necessary_regulators]

    # 5. Build the base rule parts
    parts = []
    if base_pos:
        parts.append(base_pos[0] if len(base_pos) == 1 else f"({ ' | '.join(base_pos) })")
    for n in base_neg:
        parts.append(f"!{n}")

    # 6. Append necessary regulators strictly as independent '&' terms
    for p in pos:
        if p in necessary_regulators:
            parts.append(p)
    for n in neg:
        if n in necessary_regulators:
            parts.append(f"!{n}")

    # 7. Join parts safely to avoid leading or trailing '&' symbols
    rule = " & ".join(parts) if parts else "1"
    return target, rule

In [6]:
rules = df.groupby("Target").apply(build_rule, constraints=CONSTRAINTS, ambiguous_as=AMBIGUOUS_MODE)

lines = [f"{t}, {r}" for t, r in rules]

with open(INTERMEDIATE_FILE, "w") as f:
    f.write("\n".join(lines))

print(f"Saved to {INTERMEDIATE_FILE}")

Saved to Tcell_merged_raw.bnet


In [7]:
primes = bnet_file2primes(INTERMEDIATE_FILE)
print("Loaded primes from bnet file.")

with open(OUTPUT_CACHE, "wb") as f:
    pickle.dump(primes, f)

print(f"Saved primes to {OUTPUT_CACHE}")

Loaded primes from bnet file.
Saved primes to Tcell_primes.pkl


In [8]:
bnet = primes2bnet(primes)

print("Converted primes to bnet.")
print(bnet)

with open(OUTPUT_FILE, "w") as f:
    f.write(bnet)

print(f"Saved to {OUTPUT_FILE}")

Converted primes to bnet.
APC,            APC
BCL6,           STAT4&!STAT5&!TBET&!TGFB | STAT3&!STAT5&!TBET&!TGFB | STAT1&!STAT5&!TBET&!TGFB
CD28,           APC
CD4,            !RUNX3&THPOK | NOTCH1&!RUNX3 | CD4&!RUNX3
CD8,            RUNX3&!TCR&!THPOK | NOTCH1&!TCR&!THPOK | CD8&!TCR&!THPOK
CMAF,           TGFBR | STAT3
DLL1,           DLL1
EOMES,          TBET | RUNX3 | IL27R
FOXP3,          !GATA3&!RORGT&!STAT1&!STAT3&!STAT6&!TBET&TGFBR | !GATA3&!RORGT&!STAT1&!STAT3&STAT5&!STAT6&!TBET | !GATA3&!RORGT&SMAD3&!STAT1&!STAT3&!STAT6&!TBET | !GATA3&!RORGT&SMAD2&!STAT1&!STAT3&!STAT6&!TBET | !GATA3&NFAT&!RORGT&!STAT1&!STAT3&!STAT6&!TBET | FOXP3&!GATA3&!RORGT&!STAT1&!STAT3&!STAT6&!TBET
GATA3,          !BCL6&!IL29R&!PU1&!RORGT&STAT6&!TBET&!TGFB_e | !BCL6&!IL29R&!PU1&!RORGT&STAT5&!TBET&!TGFB_e | !BCL6&!IL29R&NFAT&!PU1&!RORGT&!TBET&!TGFB_e | !BCL6&IL25R&!IL29R&!PU1&!RORGT&!TBET&!TGFB_e | !BCL6&GATA3&!IL29R&!PU1&!RORGT&!TBET&!TGFB_e
GZMB,           EOMES
IFNAR,          IFNB_e | IFNA_e
IFNA_e,    